# YoloExample notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/YoloExample/yolo_example_notebook.ipynb)

This notebook detects pedestrians with **YOLOv8n** on the Penn-Fudan dataset (170 street photos, 423 people). What the project does, and what every metric means, is in its [README](README.md). Every project has its own notebook like this one, in its folder.

Before running anything, the notebook opens up YOLO:

- **YOLO labels** (section 3): the label format, and how Penn-Fudan's masks are converted to it.
- **The YOLOv8 structure** (section 4): backbone, neck, and head, layer by layer.
- **Input and output** (section 5): what goes into the network, what each of the 84 numbers it predicts per location means, how the box is decoded, and how non-maximum suppression turns 8,400 candidates into a few boxes.
- **Metrics** (section 6): IoU, and how mAP50, mAP75, and mAP50-95 are built from it.

Then it runs the project's **three configs** and compares them with the **zero-shot baseline**:

| Config | Strategy | Starting weights | Training |
| --- | --- | --- | --- |
| `config01.json` | Zero-shot (baseline) | YOLOv8n trained on COCO (80 classes, one is `person`) | None: only evaluated, counting only its `person` boxes |
| `config02.json` | Fine-tuned | The same COCO weights | 50 epochs on Penn-Fudan, backbone frozen, gentle SGD |
| `config03.json` | From scratch | Random weights, same architecture (`yolov8n.yaml`) | 150 epochs on Penn-Fudan |

### Choose your route

| Route | Sections to run | Then run the project with |
| --- | --- | --- |
| **Colab, from notebook cells** | 1 (setup), 2, then 3 to 6 to learn, 7 and 8 to run and compare (9 and 10 to load or finish a model) | Option A1 (`!python -m ...`) or Option B (`main(...)`) |
| **Colab, from Colab's terminal** (Colab Pro) | 1 (setup) in the notebook first | Option A2: the same commands in the terminal |
| **Locally, from this notebook** | Skip 1, start at [**2. Find the repo and the data**](#2.-Find-the-repo-and-the-data) | Option A1 or B, with the `.venv` as the notebook's kernel |
| **Locally, from your terminal** | None, no notebook needed | `python -m YoloExample --config config01.json` from the repo root, with the `.venv` activated |

In Colab, the setup cells are needed even when you train in the terminal: Google Drive can only be mounted, and Colab Secrets only read, from a notebook cell. Locally, set up the repo once first (`.venv`, requirements, and `.env`, see the main README), and open this notebook from the project folder.

In Colab, run the cells from top to bottom.

1. **Use a GPU:** *Runtime → Change runtime type → GPU*. On a Colab T4, fine-tuning takes a few minutes and training from scratch around 15; on a CPU, much longer.
2. **Mount Google Drive** so your results survive runtime resets.
3. **Set up the repo:** clone (or update) it, apply the Colab settings, and install only the missing requirements (here, `ultralytics`).
4. **Load your API keys** (optional) from Colab Secrets.
5. **Learn the format and the network** (sections 3 to 6), then **run the three configs** and compare them (sections 7 and 8).

Before a long training run, read the [pro tips](#pro-tips-keep-training-running) at the end.

## 1. Setup

> **Running locally?** Skip this section: these cells only work in Colab. Jump to [**2. Find the repo and the data**](#2.-Find-the-repo-and-the-data).

Run these cells once per session (and again after the runtime resets): mount Drive, set up the repo, change directory, and load API keys. The setup cell is safe to re-run, e.g. to get new projects with `git pull`; run the API-key cell again after it.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' requirements.txt \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done

In [ ]:
# The bash cell above cannot change the notebook's working directory, so do it here.
%cd /content/CodeCS4337Fall2026

### Load API keys (optional)

This project logs every run to Weights & Biases (W&B), which needs `WANDB_API_KEY`. Without it, runs are logged to W&B offline and can be uploaded later with the `wandb sync` command printed at the end of the run.

1. Add the keys as Colab **Secrets**: key icon in the left sidebar → *Add new secret*, named `WANDB_API_KEY`. Never type keys into a notebook cell or into `.envcolab`, which is public on GitHub.
2. Run the cell below **after the setup cells and before running the project**, whether you run it from this notebook (Option A1 or B) or from Colab's terminal (A2). Run it again whenever you re-run the setup cell.

The cell loads `.env`, then looks up every key that is still missing in your Secrets (Colab may ask you to *Grant access*). Keys it finds are added to this notebook's environment and saved to `/content/CodeCS4337Fall2026/.env`, so terminal runs find them too. That file is not in git and is deleted when the runtime resets. Values are never printed.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from google.colab import userdata

ENV_FILE = Path("/content/CodeCS4337Fall2026/.env")
KEYS = ["WANDB_API_KEY"]

if load_dotenv(ENV_FILE):
    print(f"Loaded settings from {ENV_FILE}")
else:
    print(f"No settings in {ENV_FILE}: run the setup cells first")

for key in KEYS:
    if os.getenv(key):
        print(f"{key}: found in .env")
        continue
    try:
        os.environ[key] = userdata.get(key)
    except Exception:  # noqa: BLE001 (Secret not added or access not granted)
        print(f"{key}: not found, W&B will log offline")
        continue
    with ENV_FILE.open("a") as env_file:
        env_file.write(f"\n{key}={os.environ[key]}\n")
    print(f"{key}: loaded from Colab Secrets and saved to .env")

## 2. Find the repo and the data

**Run these cells first** (in Colab and locally). The first finds the repo folder and makes the project importable; the second shows where results go; the third downloads Penn-Fudan (51 MB) and converts it to YOLO format, once.

In [ ]:
import sys
from pathlib import Path

# Colab: the clone made by the setup cell. Locally: the parent of this notebook's project folder.
IN_COLAB = "google.colab" in sys.modules
REPO = Path("/content/CodeCS4337Fall2026") if IN_COLAB else Path.cwd().parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print(f"Repo folder: {REPO}")

In [ ]:
import os

from dotenv import load_dotenv

# Colab: the copy of .envcolab made by the setup cell. Locally: your own .env.
load_dotenv(REPO / ".env")
!echo "OUTPUT_DIR=$OUTPUT_DIR"
output_dir = REPO / os.getenv("OUTPUT_DIR", "runs")
print(f"Runs of this project are saved in: {output_dir / 'YoloExample'}")

`OUTPUT_DIR` is the folder where **every run of this project is saved**: weights, metrics, plots, and `runs_summary.csv`. In Colab it is in your Google Drive, `/content/drive/MyDrive/CodeCS4337Fall2026/runs` (from `.envcolab`), so your results survive runtime resets; locally it is the repo's `runs/` folder. Each run gets its own folder, `<OUTPUT_DIR>/YoloExample/<config>/<timestamp>/`.

In [ ]:
from YoloExample.dataloaders import prepare_pennfudan

data_dir = REPO / os.getenv("DATA_DIR", "data")
data_yaml = prepare_pennfudan(data_dir)
dataset = data_yaml.parent
print(f"Dataset in YOLO format: {dataset}\n")
print(data_yaml.read_text())
for split in ["train", "val", "test"]:
    images = sorted((dataset / "images" / split).glob("*.png"))
    labels = sorted((dataset / "labels" / split).glob("*.txt"))
    people = sum(len(f.read_text().splitlines()) for f in labels)
    print(
        f"{split:>5}: {len(images):3} images, {len(labels):3} label files, {people:3} people"
    )

## 3. YOLO labels: from masks to boxes

### The YOLO label format

Ultralytics reads one **text file per image**, with the same name (`FudanPed00001.png` → `FudanPed00001.txt`), in a `labels/` folder next to `images/`. Each line is **one object**:

```text
class x_center y_center width height
0 0.535714 0.534483 0.247126 0.752874
```

| Field | Meaning |
| --- | --- |
| `class` | Index of the class in the dataset's `names`, starting at 0. Penn-Fudan has one class: `0` = `person`. |
| `x_center`, `y_center` | Center of the box, divided by the image width and height: always between 0 and 1. |
| `width`, `height` | Size of the box, divided by the image width and height. |

Because the numbers are **normalized**, the labels stay valid when the image is resized, which YOLO does all the time (letterboxing, multi-scale augmentation). An image without people gets an empty (or no) label file: it is a background image, still useful as a negative example.

Other datasets store boxes differently; converting means getting this right:

| Format | Box | Units |
| --- | --- | --- |
| Pascal VOC | `x_min, y_min, x_max, y_max` (corners) | Pixels |
| COCO | `x_min, y_min, width, height` (top-left corner and size) | Pixels |
| **YOLO** | `x_center, y_center, width, height` | **Fraction of the image size** |

From pixel corners on a `W × H` image:

$$
x_\text{center} = \frac{x_\min + x_\max}{2W},\quad
y_\text{center} = \frac{y_\min + y_\max}{2H},\quad
w = \frac{x_\max - x_\min}{W},\quad
h = \frac{y_\max - y_\min}{H}
$$

The dataset itself is described by a small YAML file (printed above): the dataset folder (`path`), the image folder of each split (Ultralytics finds the labels by replacing `images` with `labels` in the path), and the class `names`.

### Converting Penn-Fudan

Penn-Fudan does not come with boxes in this format. Each image has an **instance mask**, `PedMasks/<name>_mask.png`: a picture of the same size where background pixels are 0 and the pixels of the k-th person have the value k. The conversion (`convert_to_yolo` in `dataloaders/pennfudan.py`) does, for every image:

1. Find each person `k` in the mask, and the smallest box around its pixels (`mask_to_boxes`).
2. Convert the box to YOLO's normalized numbers (`box_to_yolo`), and write one line per person.
3. Copy the image and its label file to the split it was drawn for: 120 training, 25 validation, 25 test images, shuffled with a fixed seed so every run uses the same split.

The cell below does steps 1 and 2 for one image, by hand.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from PIL import Image

from YoloExample.dataloaders import box_to_yolo, mask_to_boxes

name = "FudanPed00001"
raw = data_dir / "PennFudanPed"
image = Image.open(raw / "PNGImages" / f"{name}.png").convert("RGB")
mask = np.array(Image.open(raw / "PedMasks" / f"{name}_mask.png"))
width, height = image.size
print(
    f"Image {width}x{height}; values in the mask: {np.unique(mask).tolist()} (0 = background)"
)

boxes = mask_to_boxes(mask)
print("\nperson  pixel box (x_min, y_min, x_max, y_max)  ->  YOLO line")
for k, box in enumerate(boxes, start=1):
    cx, cy, w, h = box_to_yolo(box, width, height)
    print(f"{k:>6}  {box!s:<38}  ->  0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
axes[0].imshow(image)
axes[0].set_title("Image")
axes[1].imshow(mask, cmap="tab10", interpolation="nearest")
axes[1].set_title("Instance mask, with the boxes found in it")
for k, (x0, y0, x1, y1) in enumerate(boxes, start=1):
    axes[1].add_patch(
        patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="red", lw=2)
    )
    axes[1].text(x0, y0 - 5, f"person {k}", color="red")
for ax in axes:
    ax.axis("off")
plt.show()

**Always check converted labels by drawing them back.** A swapped width and height, or corners written where centers are expected, still trains, just badly, and nothing warns you. The cell below reads label files written by the conversion, converts them back to pixels (`yolo_to_box`), and draws them on their images.

In [ ]:
from YoloExample.dataloaders import read_yolo_labels, yolo_to_box

files = sorted((dataset / "images" / "train").glob("*.png"))[:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), layout="constrained")
for ax, file in zip(axes, files):
    picture = Image.open(file)
    label_file = dataset / "labels" / "train" / f"{file.stem}.txt"
    ax.imshow(picture)
    for cls, yolo_box in read_yolo_labels(label_file):
        x0, y0, x1, y1 = yolo_to_box(yolo_box, *picture.size)
        ax.add_patch(
            patches.Rectangle(
                (x0, y0), x1 - x0, y1 - y0, fill=False, color="lime", lw=2
            )
        )
    ax.set_title(f"{file.stem}: {len(read_yolo_labels(label_file))} people", fontsize=9)
    ax.axis("off")
plt.show()
print(f"{label_file.name}:\n{label_file.read_text()}")

## 4. The YOLOv8 structure

YOLO ("You Only Look Once") is a **one-stage detector**: a single pass through one network predicts all boxes and classes at once, which is why it runs in a few milliseconds per image. Two-stage detectors such as Faster R-CNN first propose regions, then classify each one.

YOLOv8 has three parts:

```text
 image 3x640x640
      │
 ┌────▼─────────────────────────────┐
 │ BACKBONE  (layers 0-9)           │  extracts features, halving the size 5 times
 │ Conv, Conv, C2f, Conv, C2f, ...  │
 │                                  │──► P3: 64 x 80 x 80   (stride 8)
 │                                  │──► P4: 128 x 40 x 40  (stride 16)
 │ ..., SPPF                        │──► P5: 256 x 20 x 20  (stride 32)
 └────┬─────────────────────────────┘
 ┌────▼─────────────────────────────┐
 │ NECK  (layers 10-21), PAN-FPN    │  mixes the three scales:
 │ top-down: upsample + concat      │  deep "what" information goes down to the large maps,
 │ bottom-up: conv + concat         │  precise "where" information goes up to the small maps
 └────┬───────────┬───────────┬─────┘
 ┌────▼───────────▼───────────▼─────┐
 │ HEAD  (layer 22), Detect         │  for every location of the three maps:
 │ box branch + class branch        │  4 box numbers + 80 class scores
 └──────────────────────────────────┘
      │
 predictions 84 x 8400 ──► confidence threshold + NMS ──► final boxes
```

The **`n`** in YOLOv8n is the smallest of five sizes (n, s, m, l, x). They share this layout; the model file scales how many times blocks are repeated (`depth_multiple`) and how many channels they have (`width_multiple`). YOLOv8n has about 3.2 million parameters, YOLOv8x 68 million.

The cell below loads YOLOv8n with its COCO weights (downloaded once, 6 MB, to `<DATA_DIR>/weights/`), prints the model file, and passes a 640 × 640 image through it to list every layer with its output shape.

In [ ]:
import torch
from ultralytics import YOLO

from YoloExample.models import weights_path

yolo = YOLO(weights_path("yolov8n.pt"))
net = yolo.model.eval()
print("Model file (yolov8.yaml, scaled to n):")
print(
    f"  depth_multiple={net.yaml['depth_multiple']}, width_multiple={net.yaml['width_multiple']}"
)
for part in ["backbone", "head"]:
    print(f"  {part}:")
    for row in net.yaml[part]:
        print(f"    {row}")

outputs = {}
hooks = [
    layer.register_forward_hook(lambda m, i, o, k=k: outputs.__setitem__(k, o))
    for k, layer in enumerate(net.model)
]
with torch.no_grad():
    net(torch.zeros(1, 3, 640, 640))
for hook in hooks:
    hook.remove()

print(f"\n{'layer':>5}  {'from':>12}  {'module':<9} {'params':>9}  output shape")
for k, layer in enumerate(net.model):
    out = outputs[k]
    shape = tuple(out.shape) if torch.is_tensor(out) else "see section 5"
    params = sum(p.numel() for p in layer.parameters())
    print(f"{k:>5}  {layer.f!s:>12}  {type(layer).__name__:<9} {params:>9,}  {shape}")
print(f"\nTotal: {sum(p.numel() for p in net.parameters()):,} parameters")

How to read the table: `from` is where a layer takes its input (`-1` is the previous layer; a list means several layers are concatenated), and the output shape is `(batch, channels, height, width)`. Every Conv with stride 2 halves height and width, and the channels grow as the maps get smaller.

| Module | What it does |
| --- | --- |
| **Conv** | Convolution + batch normalization + SiLU activation. With stride 2, it halves the feature map (layers 0, 1, 3, 5, 7 go from 640 to 320, 160, 80, 40, 20). |
| **C2f** | The main feature block ("CSP bottleneck with 2 convolutions, faster"). It splits its channels in two, sends one half through a chain of small residual bottlenecks, and concatenates the output of *every* bottleneck with the untouched half before a final 1×1 convolution. Many gradient paths, few parameters. |
| **SPPF** | Spatial Pyramid Pooling, Fast: three 5×5 max-poolings in a row, concatenated. Each 20 × 20 cell then sees context from a large part of the image, at almost no cost. |
| **Upsample** | Doubles height and width (nearest neighbor), so a deep, small map can be combined with a larger, earlier one. |
| **Concat** | Stacks two maps of the same size along the channels: e.g. layer 11 = upsampled layer 10 (256 channels) + backbone layer 6 (128) = 384 channels. |
| **Detect** | The head. For each of the three maps, two separate small branches: one predicts the box, the other the class scores (a *decoupled* head). |

Three feature maps leave the neck and enter the head: layer 15 (P3, 80 × 80, stride 8: each cell covers 8 × 8 input pixels, for small people far away), layer 18 (P4, 40 × 40, stride 16), and layer 21 (P5, 20 × 20, stride 32, for large people close by). Almost a third of the parameters (about 0.9 million of 3.2) are in the head; with one class instead of 80, the class branch shrinks and the model has 3.0 million.

## 5. Input and output

### Input: what the network sees

The network takes a batch of **square RGB images, 640 × 640 pixels, with values between 0 and 1**: a tensor of shape `(batch, 3, 640, 640)`. Photos are not square, so Ultralytics **letterboxes** them:

1. Resize the image so that its longer side is 640, keeping its aspect ratio (people are not stretched).
2. Pad the shorter side with gray (value 114) up to 640, equally on both sides.
3. Reorder from `height × width × channels` to `channels × height × width`, divide by 255, and add the batch dimension.

The boxes it predicts are in the coordinates of this 640 × 640 letterboxed image; Ultralytics undoes the resize and padding to report them in the original image's pixels. (During validation it pads only up to a multiple of 32, which saves time; the idea is the same.)

In [ ]:
from ultralytics.data.augment import LetterBox

test_file = min((dataset / "images" / "test").glob("*.png"))
original = np.array(Image.open(test_file).convert("RGB"))
letterboxed = LetterBox((640, 640), auto=False)(image=original)
x = torch.from_numpy(letterboxed).permute(2, 0, 1)[None].float() / 255

print(f"Original image:  {original.shape} (height, width, channels), values 0-255")
print(f"Letterboxed:     {letterboxed.shape}")
print(
    f"Network input x: {tuple(x.shape)} (batch, channels, height, width), "
    f"{x.dtype}, values {x.min():.2f} to {x.max():.2f}"
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), layout="constrained")
axes[0].imshow(original)
axes[0].set_title(f"Original {original.shape[1]}x{original.shape[0]}")
axes[1].imshow(letterboxed)
axes[1].set_title("Letterboxed 640x640: resized, then padded with gray")
plt.show()

### Output: 8,400 candidate boxes, 84 numbers each

In evaluation mode, the network returns a tensor `y` of shape **`(batch, 84, 8400)`**: 8,400 candidate boxes, each described by a vector of 84 numbers. (It also returns the raw values behind `y`, used below.)

**Why 8,400?** Every cell of the three feature maps predicts one box:

$$
\underbrace{80 \times 80}_{\text{P3, stride 8}} + \underbrace{40 \times 40}_{\text{P4, stride 16}} + \underbrace{20 \times 20}_{\text{P5, stride 32}} = 6400 + 1600 + 400 = 8400
$$

Columns 0 to 6,399 come from P3 (row by row), 6,400 to 7,999 from P4, and 8,000 to 8,399 from P5. The center of a cell is called its **anchor point**. YOLOv8 is **anchor-free**: older YOLOs started from a few preset box shapes per cell ("anchor boxes"); YOLOv8 directly predicts how far the box extends from the anchor point.

**The 84 components of each vector:**

| Index | Name | Meaning |
| --- | --- | --- |
| 0 | `cx` | x of the box center, in pixels of the 640 × 640 input |
| 1 | `cy` | y of the box center, in pixels |
| 2 | `w` | box width, in pixels |
| 3 | `h` | box height, in pixels |
| 4 | score of class 0 | probability that the box contains a `person` (COCO class 0) |
| 5 | score of class 1 | `bicycle` |
| ... | ... | ... |
| 83 | score of class 79 | `toothbrush` |

- The box is `(cx, cy, w, h)`, like a YOLO label, but in **pixels** of the input, not normalized.
- The 80 class scores are each between 0 and 1, computed **independently** (a sigmoid per class, not a softmax): they need not add up to 1, and a box can score high for two classes.
- There is **no objectness score.** YOLOv5 predicted 85 numbers: 4 + 1 "is there an object here?" + 80. YOLOv8 drops it; a box's **confidence** is simply its highest class score.
- A model trained on Penn-Fudan has one class, so its vectors have **5** components: `cx, cy, w, h, person`.

The cell below runs the network on the letterboxed image and prints the vector of the anchor point most confident that it sees a person.

In [ ]:
from ultralytics.utils.torch_utils import get_flops

with torch.no_grad():
    y, raw = net(x)
print(f"y: {tuple(y.shape)}  (batch, 4 box + 80 class numbers, candidate boxes)")

LEVELS = [(80, 8), (40, 16), (20, 32)]  # (grid size, stride) of P3, P4, P5


def locate(index):
    # Which feature map, grid cell, and stride a column of y comes from.
    start = 0
    for level, (size, stride) in enumerate(LEVELS, start=3):
        if index < start + size * size:
            row, col = divmod(index - start, size)
            return level, row, col, stride
        start += size * size


best = int(y[0, 4].argmax())
vector = y[0, :, best]
level, row, col, stride = locate(best)
print(
    f"\nColumn {best} of y: map P{level} (stride {stride}), grid row {row}, column {col}"
)
print(
    f"anchor point: ({(col + 0.5) * stride:.0f}, {(row + 0.5) * stride:.0f}) pixels\n"
)
for i, name in enumerate(["cx", "cy", "w", "h"]):
    print(f"  [{i}] {name:<3} = {vector[i]:7.1f} pixels")
scores = vector[4:]
print("  [4..83] 80 class scores; the 5 highest:")
for cls in scores.argsort(descending=True)[:5].tolist():
    print(f"      [{cls + 4:>2}] {yolo.names[cls]:<12} {scores[cls]:.4f}")
print(
    f"  sum of the 80 scores: {scores.sum():.3f} (independent sigmoids, not a softmax)"
)
print(
    f"\nYOLOv8n: {sum(p.numel() for p in net.parameters()):,} parameters, "
    f"{get_flops(net, 640):.1f} GFLOPs per 640x640 image"
)

### Inside the head: how the box is predicted (DFL)

The head does not predict `cx, cy, w, h` directly. For each anchor point, the box branch predicts the distance from the anchor point to the box's **left, top, right, and bottom** sides. And not as 4 numbers: as **4 probability distributions over 16 possible distances** (0 to 15, in units of the stride). The distance used is the **expected value** of each distribution. This is called **Distribution Focal Loss (DFL)**, after the loss that trains it. A wide distribution means the network is unsure where that edge is, e.g. a person partly hidden behind another.

So each anchor point really gets **144 raw numbers**: 64 for the box (4 sides × 16 bins, `reg_max = 16`) and 80 class logits. They are in `raw`:

| Key | Shape | Content |
| --- | --- | --- |
| `raw["boxes"]` | `(1, 64, 8400)` | the 4 × 16 distance logits per anchor point |
| `raw["scores"]` | `(1, 80, 8400)` | class logits; `sigmoid` turns them into the scores in `y` |
| `raw["feats"]` | 3 maps | P3 `(1, 64, 80, 80)`, P4 `(1, 128, 40, 40)`, P5 `(1, 256, 20, 20)`: the neck's outputs |

Decoding one anchor point by hand, with $s$ the stride and $(a_x, a_y)$ the anchor point in grid units (cell column + 0.5, row + 0.5):

$$
d_\text{side} = \sum_{i=0}^{15} i \cdot \operatorname{softmax}(\text{logits}_\text{side})_i,
\qquad
x_1 = a_x - d_\text{left},\; y_1 = a_y - d_\text{top},\; x_2 = a_x + d_\text{right},\; y_2 = a_y + d_\text{bottom}
$$

$$
c_x = s\,\frac{x_1 + x_2}{2},\quad c_y = s\,\frac{y_1 + y_2}{2},\quad w = s\,(x_2 - x_1),\quad h = s\,(y_2 - y_1)
$$

The cell below does exactly this and compares with what the network returned in `y`.

In [ ]:
logits = raw["boxes"][0, :, best].view(4, 16)  # left, top, right, bottom
bins = logits.softmax(dim=1)
distances = (bins * torch.arange(16)).sum(dim=1)
ax_, ay_ = col + 0.5, row + 0.5
x1, y1 = ax_ - distances[0], ay_ - distances[1]
x2, y2 = ax_ + distances[2], ay_ + distances[3]
decoded = torch.stack([(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]) * stride

print(
    "distances (grid units): left {:.2f}, top {:.2f}, right {:.2f}, bottom {:.2f}".format(
        *distances
    )
)
print(f"decoded by hand: cx, cy, w, h = {[round(v, 1) for v in decoded.tolist()]}")
print(f"network's y:     cx, cy, w, h = {[round(v, 1) for v in vector[:4].tolist()]}")
same = torch.allclose(raw["scores"][0, :, best].sigmoid(), vector[4:], atol=1e-5)
print(f"sigmoid(raw class logits) == y[4:]: {same}")

fig, axes = plt.subplots(1, 4, figsize=(15, 2.8), layout="constrained", sharey=True)
for axis, side, p, d in zip(axes, ["left", "top", "right", "bottom"], bins, distances):
    axis.bar(range(16), p.numpy())
    axis.axvline(d, color="red", label=f"expected = {d:.2f}")
    axis.set_title(f"distance to the {side} side")
    axis.set_xlabel("bin (x stride pixels)")
    axis.legend(fontsize=8)
axes[0].set_ylabel("probability")
plt.show()

### From 8,400 candidates to a few boxes

Neighboring anchor points all see the same person, so many candidates overlap. Post-processing keeps the good ones:

1. **Confidence threshold:** drop candidates whose best class score is below `conf` (0.25 when predicting, 0.001 when computing mAP, which needs the whole precision-recall curve).
2. **Non-maximum suppression (NMS):** sort the rest by confidence; keep the best box, delete every other box of the same class that overlaps it with IoU above `iou` (0.7); repeat with the next remaining box.
3. Convert to corners and undo the letterbox: the `Results` object holds `boxes.xyxy` (pixels of the original image), `boxes.conf`, and `boxes.cls`.

The COCO model knows 80 classes, so on a street photo it also finds cars, handbags, and bicycles. That is why the zero-shot baseline (config01) passes `classes=[0]`: only its `person` boxes are compared with Penn-Fudan's labels.

In [ ]:
from YoloExample.dataloaders import read_yolo_labels, yolo_to_box

candidates = int((y[0, 4:].max(dim=0).values > 0.25).sum())
# A second copy: predict() moves its model to the GPU and fuses layers, net stays as it is.
detector = YOLO(weights_path("yolov8n.pt"))
result = detector.predict(original, conf=0.25, verbose=False)[0].cpu()
print(
    f"{candidates} of 8400 candidates have a confidence above 0.25; after NMS: {len(result.boxes)} boxes\n"
)
print(f"{'class':<12} {'conf':>5}  box (x1, y1, x2, y2) in original pixels")
for xyxy, conf, cls in zip(result.boxes.xyxy, result.boxes.conf, result.boxes.cls):
    print(
        f"{yolo.names[int(cls)]:<12} {conf:5.2f}  {[round(v) for v in xyxy.tolist()]}"
    )

height, width = original.shape[:2]
truth = [
    yolo_to_box(b, width, height)
    for _, b in read_yolo_labels(dataset / "labels" / "test" / f"{test_file.stem}.txt")
]
fig, ax = plt.subplots(figsize=(9, 6.5), layout="constrained")
ax.imshow(original)
for x0, y0, x1, y1 in truth:
    ax.add_patch(
        patches.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="lime", lw=3)
    )
for (x0, y0, x1, y1), conf, cls in zip(
    result.boxes.xyxy.tolist(), result.boxes.conf, result.boxes.cls
):
    color = "red" if int(cls) == 0 else "orange"
    ax.add_patch(
        patches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0, fill=False, color=color, lw=1.5, ls="--"
        )
    )
    ax.text(x0, y0 - 3, f"{yolo.names[int(cls)]} {conf:.2f}", color=color, fontsize=8)
ax.set_title(
    "Green: Penn-Fudan labels. Dashed: zero-shot predictions (red: person, orange: other COCO classes)"
)
ax.axis("off")
plt.show()

### What training changes

When the model is trained (config02 and config03), each labeled person is assigned to the anchor points that best predict it (the *task-aligned assigner*, which looks at both class score and box overlap). Three losses are minimized, and logged to W&B every epoch as `train/...` and `val/...`:

| Loss | What it measures |
| --- | --- |
| `box_loss` | How badly the predicted box overlaps the true box (CIoU: 1 − IoU, plus penalties for center distance and aspect ratio) |
| `cls_loss` | Binary cross-entropy of the class scores: high for persons, low elsewhere |
| `dfl_loss` | Distribution Focal Loss: pushes each side's 16-bin distribution toward the true distance |

When fine-tuning from the COCO model, the 80-class score layer is replaced by a fresh 1-class layer; everything else starts from COCO's weights.

## 6. Metrics: IoU, precision, recall, and mAP

All detection metrics start from **IoU** (intersection over union): the area where a predicted box and a true box overlap, divided by the area they cover together. 1 is a perfect fit, 0 no overlap. A prediction is a **true positive** if its IoU with a still-unmatched true box is at least a threshold, otherwise a **false positive**; unmatched true boxes are **false negatives** (missed people).

From these, **precision** (how many detections are right) and **recall** (how many people are found). Sweeping the confidence threshold traces the **precision-recall curve**; its area is the **average precision (AP)**. Then:

| Metric | Meaning |
| --- | --- |
| **mAP50** | AP with IoU ≥ 0.5: a loose fit counts |
| **mAP75** | AP with IoU ≥ 0.75: the box must be tight |
| **mAP50-95** | the average of AP at IoU 0.50, 0.55, ..., 0.95 (10 thresholds), written mAP@0.5:0.05:0.95: the main COCO metric, rewarding both finding people and fitting them precisely |

With one class, "mean" AP is just the AP of `person`. The [README](README.md#metrics) explains each metric in detail, with how to read the plots in W&B. The cell below computes the IoU of each zero-shot person prediction with its best-matching true box, and the thresholds it passes.

In [ ]:
from torchvision.ops import box_iou

people = result.boxes.xyxy[result.boxes.cls == 0]
ious = box_iou(people, torch.tensor(truth, dtype=torch.float32))
thresholds = np.arange(0.5, 0.96, 0.05)
print(
    f"{'prediction':>10}  {'best IoU':>8}  thresholds passed (of 0.50, 0.55, ..., 0.95)"
)
for i, row_ious in enumerate(ious):
    best_iou = float(row_ious.max())
    passed = int((best_iou >= thresholds).sum())
    print(f"{i:>10}  {best_iou:8.3f}  {passed}/10")

A prediction with IoU 0.80 counts as correct for 7 of the 10 thresholds (0.50 to 0.80), and as a false positive for the other three: mAP50-95 gives it partial credit, mAP50 full credit.

Predictions with a very low best IoU are often **not mistakes of the model**: Penn-Fudan does not label every person. People mostly hidden behind someone else, or small in the background, often have no box, so finding them counts as a false positive. Fine-tuning teaches the model these labeling habits, which is part of why it beats the zero-shot baseline.

## 7. Run the project (choose one option)

Both options run exactly the same thing and save results to the same folder (Google Drive in Colab, `runs/` locally). Pass a config name from the project's `configs/` folder: `config01.json` (zero-shot baseline, no training, under a minute), `config02.json` (fine-tuned), and `config03.json` (from scratch). To try your own settings, copy one of them in the file browser (left sidebar, `CodeCS4337Fall2026/YoloExample/configs/`), e.g. to `config04.json`, edit it, and pass its name.

Run **config01 first**: every later run compares itself with the newest zero-shot run (`gain_vs_zero_shot`).

| Option | How | When to use it |
| --- | --- | --- |
| **A. Terminal command** | `!` in a notebook cell, or Colab's terminal (Colab Pro) | Same command as on your own machine |
| **B. From Python** | `from YoloExample import main` | Simple, everything stays in the notebook |

### Option A: Run as a terminal command

**A1. From a notebook cell (everyone):** a line starting with `!` is sent to the shell, and `{REPO}` is replaced by the repo folder found in section 2. Locally, if it reports a missing package, the shell is not using the `.venv`'s Python: use Option B, or run the same command in your own terminal with the `.venv` activated.

In [ ]:
!cd "{REPO}" && for c in config01 config02 config03; do python -m YoloExample --config $c.json; done

**A2. From Colab's terminal (Colab Pro):** open the terminal from the **Terminal** button at the bottom left of the Colab window, then run:

```bash
cd /content/CodeCS4337Fall2026
python -m YoloExample --config config01.json
python -m YoloExample --config config02.json
python -m YoloExample --config config03.json
```

The terminal runs on the same machine as this notebook, so it uses the same files, packages, and Drive mount. **You still need to run the setup cells first**: Drive can only be mounted from a notebook cell, and Colab Secrets can only be read from a notebook cell (the API-key cell saves them to `.env`, where the terminal finds them).

### Option B: Run from Python

Each call evaluates (config01) or trains then evaluates (config02, config03) one config, on the validation and test splits, and returns the test metrics. Run a single config by keeping only its name in the list.

In [ ]:
from YoloExample import main

test_metrics = {}
for config in ["config01.json", "config02.json", "config03.json"]:
    test_metrics[config] = main(config)

## 8. Results

Each run is saved under `<OUTPUT_DIR>/YoloExample/<config>/<timestamp>/`, and each config has a `runs_summary.csv` with one row per run. The cell below compares the **newest run of each config** with the zero-shot baseline (`compare_runs`, the same table each run logs to W&B as `comparison`): `gain_vs_zero_shot` is its test mAP50-95 minus the zero-shot run's.

In [ ]:
from YoloExample.utils import compare_runs

project_runs = REPO / os.getenv("OUTPUT_DIR", "runs") / "YoloExample"
rows = compare_runs(project_runs)
columns = [
    "config",
    "strategy",
    "epochs_run",
    "test_precision",
    "test_recall",
    "test_mAP50",
    "test_mAP75",
    "test_mAP50-95",
    "gain_vs_zero_shot",
]
print("".join(f"{c:>18}" for c in columns))
for row in rows:
    values = [
        row[c]
        if c in ("config", "strategy", "epochs_run")
        else (
            ""
            if row.get(c) is None
            else f"{float(row[c]):+.4f}"
            if c.startswith("gain")
            else f"{float(row[c]):.4f}"
        )
        for c in columns
    ]
    print("".join(f"{v:>18}" for v in values))

if rows:
    metrics = ["test_mAP50", "test_mAP75", "test_mAP50-95"]
    fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
    bar = 0.8 / len(rows)
    for i, row in enumerate(rows):
        values = [float(row[m]) for m in metrics]
        ax.bar(
            np.arange(3) + i * bar,
            values,
            bar,
            label=f"{row['config']} ({row['strategy']})",
        )
    ax.set_xticks(
        np.arange(3) + bar * (len(rows) - 1) / 2, ["mAP50", "mAP75", "mAP50-95"]
    )
    ax.set_ylim(0, 1)
    ax.set_ylabel("test score")
    ax.legend()
    plt.show()

**What to expect** (one run of each, seed 42; your numbers will differ a little):

| Config | Test mAP50 | Test mAP75 | Test mAP50-95 | Gain vs zero-shot |
| --- | --- | --- | --- | --- |
| config01, zero-shot | 0.950 | 0.924 | 0.826 | 0 |
| config02, fine-tuned | 0.957 | 0.926 | 0.854 | +0.027 |
| config03, from scratch | 0.866 | 0.601 | 0.521 | −0.306 |

- The COCO model has seen many thousands of people, so the **zero-shot baseline is strong**.
- **Fine-tuning gently** (config02: frozen backbone, SGD with a small learning rate) adapts it to how Penn-Fudan draws its boxes and beats the baseline, with higher precision (fewer false alarms). Fine-tuning too aggressively, e.g. `"optimizer": "auto"` (AdamW with a 2 times larger learning rate) and nothing frozen, ended *below* the baseline (about 0.77): it overwrites what the model learned on COCO with what 120 images can teach. Try it in a copy of config02.
- **From scratch** (config03), the model learns to find people (mAP50 0.87) but not to fit them tightly (mAP75 0.60): 120 images are not enough to learn what COCO's 118,000 taught. This gap between mAP50 and mAP75 is exactly what mAP50-95 captures.
- The test split has only 25 images (65 people): differences of 0.01 to 0.02 are within run-to-run noise.

### In W&B

Open the W&B project `YoloExample` (the link is printed at the end of each run). The runs are grouped by config.

- **Learning curves** (config02 and config03): `train/box_loss`, `train/cls_loss`, `train/dfl_loss`, the same losses on `val/`, and `val/precision`, `val/recall`, `val/mAP50`, `val/mAP50-95`, all against `epoch`.
- **Final scores** in each run's summary, for `val/` and `test/`: `precision`, `recall`, `f1`, `mAP50`, `mAP75`, `mAP50-95`, `inference_ms`. Add them as columns of the runs table to compare all runs.
- **Tables:** `results` (this run, both splits) and `comparison` (newest run of every config, with `gain_vs_zero_shot`).
- `ap_per_iou_threshold`: AP at each IoU threshold from 0.50 to 0.95; mAP50-95 is its average.
- **Plots** from Ultralytics: `<split>/pr_curve`, `f1_curve`, `precision_curve`, `recall_curve`, `confusion_matrix`.
- `<split>_predictions`: every image with its true (ground truth) and predicted boxes, toggled in the image panel; sort by `missed` or `false_alarms`. `<split>_hardest_examples` shows the images with the most mistakes.

## 9. Load a trained model

Running a config again trains a **new** model. To use a model you already trained (after a runtime reset, to test it again, or to try it on your own images), load it with `load_model`: it finds the weights under `<OUTPUT_DIR>/YoloExample/` and returns an Ultralytics `YOLO` model, ready for `predict` or `val`. Nothing is trained.

**Before this section:** you need at least one run of the config (in Colab: trained with Drive mounted). Then, in Colab, run the setup cells of section 1; in Colab and locally, run the first cell of section 2.

| Call | What it loads |
| --- | --- |
| `load_model("config02")` | The newest run of `config02`, its best epoch (`weights/best.pt`, highest validation mAP50-95) |
| `load_model("config02/2026-10-04_17-30-00")` | One specific run: its folder name under `YoloExample/config02/` |
| `load_model("config02", which="last")` | The last epoch instead of the best one |
| `load_model("config01")` | The zero-shot run: its config's COCO weights, since nothing was trained |

In [ ]:
from YoloExample import load_model

model = load_model("config02")

Try it on the test images: green boxes are Penn-Fudan's labels, red boxes the model's predictions with their confidence. A one-class model only predicts `person`, so no `classes` filter is needed.

In [ ]:
files = sorted((dataset / "images" / "test").glob("*.png"))[:6]
results = model.predict([str(f) for f in files], conf=0.25, verbose=False)
fig, axes = plt.subplots(2, 3, figsize=(15, 9), layout="constrained")
for ax, file, res in zip(axes.flat, files, results):
    picture = Image.open(file)
    ax.imshow(picture)
    labels = read_yolo_labels(dataset / "labels" / "test" / f"{file.stem}.txt")
    for _, b in labels:
        x0, y0, x1, y1 = yolo_to_box(b, *picture.size)
        ax.add_patch(
            patches.Rectangle(
                (x0, y0), x1 - x0, y1 - y0, fill=False, color="lime", lw=3
            )
        )
    for (x0, y0, x1, y1), conf in zip(res.boxes.xyxy.tolist(), res.boxes.conf.tolist()):
        ax.add_patch(
            patches.Rectangle(
                (x0, y0), x1 - x0, y1 - y0, fill=False, color="red", lw=1.5, ls="--"
            )
        )
        ax.text(x0, y0 - 3, f"{conf:.2f}", color="red", fontsize=8)
    ax.set_title(
        f"{file.stem}: {len(labels)} people, {len(res.boxes)} detected", fontsize=9
    )
    ax.axis("off")
plt.show()

metrics = model.val(data=str(data_yaml), split="test", plots=False, verbose=False)
print(
    f"test: mAP50 {metrics.box.map50:.4f}, mAP75 {metrics.box.map75:.4f}, mAP50-95 {metrics.box.map:.4f}"
)

The test scores should match the run's `test_mAP50-95` in `runs_summary.csv` when you loaded the best weights (to within rounding; evaluate `load_model("config01")` with `classes=[0]` to reproduce the zero-shot score).

## 10. Finish an interrupted run

If a training run is cut off (a Colab disconnect, a crash, or *Interrupt*), finish it with `resume_from`. Ultralytics saves `weights/last.pt` after every epoch, with the weights, the optimizer state, and the epoch reached; resuming continues **in the same run folder**, from the next epoch, up to the run's original number of epochs. The evaluation, W&B report, and `runs_summary.csv` row are then made as for a normal run, in a new W&B run named `<timestamp>-resumed`.

| Call | What it finishes |
| --- | --- |
| `main("config02.json", resume_from="config02")` | The newest run of `config02` |
| `main("config02.json", resume_from="config02/2026-10-04_17-30-00")` | One specific run |

A run that already **finished** (all epochs, or stopped early by `patience`) cannot be resumed: `main` stops with a message saying so, before starting anything. To train longer, edit `"epochs"` in a copy of the config and start a new run, e.g. from the earlier run's weights by setting `"weights"` to its `best.pt` path.

The same from a terminal: `python -m YoloExample --config config02.json --resume-from config02`.

In [ ]:
from YoloExample import main

# Only works for a run that was interrupted before its last epoch.
main("config02.json", resume_from="config02")

## Pro tips: keep training running

Colab disconnects runtimes that look idle (no interaction for a while, often around 90 minutes) and also caps how long a runtime can run in total (around 12 hours on the free tier, longer on paid plans). The exact limits vary and are not guaranteed. When the runtime disconnects, training stops and everything under `/content/` (outside Drive) is deleted.

**Save your state in Google Drive**
- Always mount Drive *before* training. Results then go straight to Drive (`OUTPUT_DIR` in `.envcolab`), not to the temporary disk.
- Two weight files are saved to Drive *during* training, in the run's `weights/` folder: the best epoch so far (`best.pt`) and the last epoch (`last.pt`), after every epoch. If the runtime disconnects mid-run, both are kept.
- The evaluation, the W&B report, and the row in `runs_summary.csv` are made only when the run finishes. To finish an interrupted run, resume it (section 10).
- Keep your notebook in Drive too: *File → Save a copy in Drive*. Keep copies of configs you edited in Drive, since `/content/CodeCS4337Fall2026` is deleted on reset.

**Avoid idle disconnects**
- Keep the Colab browser tab open, and don't let your computer go to sleep (plug in your laptop and turn off sleep while training).
- Check on the notebook now and then (scroll, look at the progress bars). Running in the terminal does *not* prevent disconnects: the runtime is shared.
- Don't use scripts or browser extensions that fake activity (auto-clickers); they can get your Colab usage restricted.
- With Colab Pro+, *background execution* keeps the runtime alive even after you close the browser.

**Plan your runs**
- Test a new config with few epochs first (e.g. `"epochs": 2`), then start the full run.
- Run the three configs one at a time if your session is short: each `main(...)` call saves its own results, and `config03` (from scratch) takes the longest.
- If the GPU runs out of memory, lower `"batch"` in the config's `"data"` section.
- After a disconnect, re-run the setup cells (including the API-key cell), then resume the run with `resume_from` (section 10) instead of starting over.
- When you are done, free the GPU with *Runtime → Disconnect and delete runtime*. It saves your GPU quota (or compute units on paid plans) for next time.